## Fortran extension usage and development
This notebook compiles a small Fortran extension, loads it from Python with `ctypes`, and registers it with `pipic`.

The Fortran callback removes particles within n-cells of the border by setting their weight to zero. Note that this is only an example showing the capabilities of extensions in pi-PIC. Note that this implementation will not create physically accurate behavior if used in simulations, it is only an example of the capabilities of the code.

## Build the Fortran extension
The source file `fortran/ext.f90` exports a C-compatible callback named `remove_particles_near_border`.
We compile it into a shared object that Python can load directly.

In [ ]:
fortran_dir = "./fortran"
source = fortran_dir + "/ext.f90"
shared_lib = fortran_dir + "/libparticle_removal_extension.so"

# Compile the Fortran source code into a shared library.
!gfortran -shared -fPIC {source} -o {shared_lib}

## Load the library in Python
Use `ctypes` to import the shared object and obtain the function pointer address that `pipic` expects.

In [ ]:
import ctypes
import numpy as np
import pipic
from numba import cfunc, carray
from pipic import consts, types
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display, clear_output, Image

# Load the compiled Fortran library and extract the callback address.
lib = ctypes.CDLL(str(shared_lib))
handler_address = ctypes.cast(lib.remove_particles_near_border, ctypes.c_void_p).value
print(f"Loaded handler address: {handler_address}")

# Particle source for the demo simulation.
density = 1e18
@cfunc(types.add_particles)
def density_profile(r, data_double, data_int):
    return density

## Create a small simulation

In [ ]:
# simulation variables 
temperature = 1e-6 * consts.electron_mass * consts.light_velocity**2
density = 1e18
debye_length = np.sqrt(temperature / (4 * np.pi * density * consts.electron_charge**2))
plasma_period = np.sqrt(np.pi * consts.electron_mass / (density * consts.electron_charge**2))
l = 128 * debye_length
zmin, zmax = -l / 2, l / 2
ymin, ymax = -l / 2, l / 2
xmin, xmax = -l / 2, l / 2
field_amplitude = 0.01 * 4 * np.pi * (xmax - xmin) * consts.electron_charge * density
nx, ny, nz = 2, 2, 128
timestep = plasma_period / 64

dmin = - l/4
dmax = + l/4

# Define functions for initiating the simulation
@cfunc(types.add_particles)
def density_profile(r, data_double, data_int):
    return density

@cfunc(types.field_loop)
def initial_field(ind, r, E, B, data_double, data_int):
    E[2] = field_amplitude * np.sin(4*np.pi * r[2]/ (zmax-zmin)) 

# initialize simulation
sim = pipic.init(solver='fourier_boris', # using ecnergy-conserving (ec) solver
               xmin=xmin,xmax=xmax,
               ymin=ymin,ymax=ymax,
               zmin=zmin,zmax=zmax,
               nx=nx,ny=ny,nz=nz)

# add field according to initial_field
sim.field_loop(handler=initial_field.address) # setting initial field

# add particles according to density_profile
sim.add_particles(name='particle_name',
                  number= nx*ny*nz*100, # total number of particles to add 
                  density=density_profile.address,
                  charge=consts.electron_charge,
                  mass=consts.electron_mass,
                  temperature=temperature,)


# Diagnostics
field_dd = np.zeros((1, nz), dtype=np.double)  # array for saving Ez-field
@cfunc(types.field_loop)
def field_loop(ind, r, E, B, data_double, data_int):
    # read Ez in the xz plane at y=0
    data = carray(data_double, field_dd.shape, dtype=np.double)
    if ind[1] == ny // 2 and ind[0] == nx//2: 
        data[0,ind[2]] = E[2]

particle_dd = np.zeros((64, nz), dtype=np.double)  # array for saving particle (integrated) phase-space
pmin = -np.sqrt(consts.electron_mass * temperature)*10 # minimum momentum
pmax = np.sqrt(consts.electron_mass * temperature)*10 # maximum momentum
dp = (pmax - pmin) / particle_dd.shape[0] # momentum step
dz = (zmax - zmin) / particle_dd.shape[1] # position step
@cfunc(types.particle_loop)
def particle_loop(r, p, w, id, data_double, data_int):
    # save particle momentum and position
    data = carray(data_double, particle_dd.shape, dtype=np.double)
    ip = int(particle_dd.shape[0] * (p[2] - pmin) / (pmax - pmin))
    iz = int(particle_dd.shape[1] * (r[2] - zmin) / (zmax - zmin))
    if ip >= 0 and ip < particle_dd.shape[0] and iz < particle_dd.shape[1] and iz >= 0:
        data[ip, iz] += w[0] / (dz * dp) / density # normalize by dz, dp and density

print("Particles before the Fortran handler:", sim.get_number_of_particles())

In [ ]:
# Pass the thickness of the area in which particles should be removed as the number of cells from the boundary to the Fortran handler.
data_int = np.array([8], dtype=np.intc)

sim.add_handler(
    name="remove_particles_near_border",
    subject="particle_name",
    handler=handler_address,
    data_int=pipic.addressof(data_int),
)

# advance one step with the extension enabled
sim.advance(time_step=timestep, number_of_iterations=1)

In [ ]:
# Create figures and run simulation (animation)
fig, ax = plt.subplots(2, 1, constrained_layout=True)

# with removing extension
z_axis = np.linspace(zmin, zmax, nz)
Ez_plot = ax[1].plot(z_axis,field_dd[0,:])[0]
ax[1].set_ylim(-field_amplitude, field_amplitude)
zpz_plot = ax[0].imshow(particle_dd / (3/pmax) / (xmax - xmin) / (ymax - ymin),
             extent=[zmin, zmax,pmin, pmax], 
             aspect='auto', origin='lower', 
             cmap='YlOrBr',vmin=0, vmax=1,
             interpolation = 'none')

# set titles
ax[1].set_xlabel('z (cm)')
ax[0].set_ylabel('$p_z$ (cm g/s)')
ax[1].set_ylabel('$E_z$ (StatV/cm)')

# Animation loop
simulation_steps = int(2 * plasma_period / timestep)
frames = simulation_steps // 8 # number of frames to show in the animation
counter = 0

def animate(i):
    sim.advance(time_step=timestep, number_of_iterations=8,use_omp=True)

    sim.field_loop(handler=field_loop.address, data_double=pipic.addressof(field_dd), use_omp=True)
    particle_dd.fill(0)
    sim.particle_loop(name='particle_name', handler=particle_loop.address, data_double=pipic.addressof(particle_dd))
    Ez_plot.set_ydata(field_dd)
    zpz_plot.set_data(particle_dd / (5/pmax) / (xmax - xmin) / (ymax - ymin))


    global counter
    clear_output()
    if counter <= frames:
        display(HTML('<pre> Progress: ' + "{:.2f}".format(100*counter/frames) + '%</pre>'), display_id = True)
    counter += 1
    return

ani = animation.FuncAnimation(fig, animate, frames=frames, interval = 40)
html = HTML(ani.to_jshtml())
display(html)
plt.close()

In [ ]:
print("Fortran extension loaded and registered successfully.")
print("Particles after the Fortran handler:", sim.get_number_of_particles())